# ARCO-ERA5: quickstart

Open ERA5 reanalysis data directly from Google Cloud Storage, with no
download queue, no API key, and no registration. ARCO-ERA5 is a public,
cloud-optimised Zarr copy of ERA5 maintained by Google Research.

This notebook subsets one day of 2 metre temperature over the UK, inspects
the result, and plots a map. The data never hits disk unless you choose to
save it.

Full reference: [`docs/arco-era5/README.md`](../docs/arco-era5/README.md).

## Before you run this

1. No account or API key needed. The GCS bucket is public.
2. Install the Python stack:
   ```bash
   pip install xarray zarr gcsfs matplotlib cartopy netcdf4
   ```

The default config opens the store lazily, subsets to 24 hours of
2 metre temperature over the UK, and plots a single-hour map. Edit the
config cell to change variables, dates, or region.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
# Public Zarr store URL on Google Cloud Storage.
STORE_URL = (
    "gs://gcp-public-data-arco-era5/ar/"
    "full_37-1h-0p25deg-chunk-1.zarr-v3"
)

# Variable name as it appears in the AR store.
VARIABLE = "2m_temperature"

# Time range (hourly, 1979-present).
TIME_START = "2023-07-01T00:00"
TIME_END = "2023-07-01T23:00"

# Bounding box in decimal degrees. Default: UK.
LAT_NORTH, LAT_SOUTH = 55, 49
LON_WEST, LON_EAST = -8, 2

# Optional: save the subset to NetCDF. Set to None to skip.
OUTPUT_DIR = "../data/arco-era5"
OUTPUT_FILENAME = "arco_era5_quickstart.nc"
# ==================================================================

## Imports and version check

Printing library versions helps reproduce results on a different machine.
Unlike the CDS-based notebooks in this repo, there is no credential check
here because the ARCO-ERA5 bucket is public.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["xarray", "zarr", "gcsfs", "matplotlib", "cartopy", "netcdf4"]:
    try:
        print(f"{pkg:12} {version(pkg)}")
    except Exception:
        print(f"{pkg:12} (not installed)")

# Find the repo root by walking up from CWD until we see CLAUDE.md.
def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError(
        "Could not find repo root. Run this notebook from inside the "
        "climate-data-quickstart repository."
    )

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("\nNo credentials needed - ARCO-ERA5 is a public bucket.")

## Open the Zarr store

This connects to the remote store and reads coordinate metadata only. No
data bytes are transferred yet. `chunks=None` keeps data variables lazy
(backed by the Zarr store) while loading coordinates eagerly so `.sel()`
works immediately. The `token="anon"` setting tells gcsfs the bucket is
public.

In [ ]:
ds = xr.open_zarr(
    STORE_URL,
    chunks=None,
    storage_options=dict(token="anon"),
)

# Filter to the valid time range the store advertises in its metadata
ds = ds.sel(time=slice(ds.attrs["valid_time_start"], ds.attrs["valid_time_stop"]))

print(f"Variables:  {len(ds.data_vars)}")
print(f"Time range: {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}")
print(f"Latitude:   {float(ds.latitude.min()):.2f} to {float(ds.latitude.max()):.2f}")
print(f"Longitude:  {float(ds.longitude.min()):.2f} to {float(ds.longitude.max()):.2f}")
print(f"\nData variables: {list(ds.data_vars)[:10]} ...")

## Subset in space and time

Still lazy. `.sel()` defines which byte-ranges will be fetched when we
call `.load()`. ARCO-ERA5 latitude runs 90 to -90 (descending), so
`slice(LAT_NORTH, LAT_SOUTH)` gives the correct direction. Longitude is
-180 to 180, matching typical negative-west conventions.

The printed shape and memory estimate confirm the subset is small enough
to load comfortably.

In [ ]:
subset = ds[VARIABLE].sel(
    time=slice(TIME_START, TIME_END),
    latitude=slice(LAT_NORTH, LAT_SOUTH),
    longitude=slice(LON_WEST, LON_EAST),
)

print(f"Lazy subset shape: {subset.shape}")
print(f"Size if loaded:    {subset.nbytes / 1e6:.2f} MB")
print(subset)

## Materialise and plot

Now we fetch bytes from GCS. For this small UK/one-day subset the transfer
takes a few seconds. We select a single hour (12:00 UTC) for a clean 2D
map and convert from kelvin to celsius.

Optionally, the cell also saves the full 24-hour subset to NetCDF so you
have a local copy. Remove or comment out the `.to_netcdf()` line if you
prefer to work entirely in memory.

In [ ]:
# Load the full 24-hour subset into memory
subset_loaded = subset.load()

# Save to NetCDF if OUTPUT_DIR is set
if OUTPUT_DIR is not None:
    output_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
    output_path.parent.mkdir(parents=True, exist_ok=True)
    subset_loaded.to_netcdf(output_path)
    print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

# Select midday for a single 2D map
t2m_midday = subset_loaded.sel(time="2023-07-01T12:00", method="nearest")
t2m_celsius = t2m_midday - 273.15

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

t2m_celsius.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": "2 m temperature (C)"},
)

ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

ax.set_title("ARCO-ERA5: 2 m temperature, 2023-07-01 12:00 UTC")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/arco-era5/README.md`](../docs/arco-era5/README.md)
  for variables, access, gotchas, licence, and citation.
- **Pressure-level data**: the same store contains 7 variables on 37
  pressure levels. Select a level with
  `ds["temperature"].sel(level=850, ...)`.
- **Aggregate to daily**: `subset.resample(time="1D").mean()` gives daily
  means. No pre-computed daily product exists for ARCO-ERA5, so you build
  aggregates from the hourly data.
- **Full variable catalogue**: inspect `list(ds.data_vars)` after opening
  the store to see all available variables.
- **CDS ERA5 for wider coverage**: ARCO-ERA5 covers ~38 variables from
  1979. For the full 250+ variable catalogue, wave data, ensemble members,
  or pre-1979 back extension, use the CDS directly. See
  `docs/era5-single-levels/` in this repo.
- **Earth Data Hub**: an alternative streaming ERA5 source with
  authentication. See `docs/earth-data-hub/` in this repo.
- **Egress awareness**: reading large volumes from outside GCP may incur
  charges. For heavy workloads, consider running in a GCP VM in
  `us-central1`.